# E-STEER — paper-faithful experiment (Qwen3-8B or Qwen3-1.7B, chosen by your GPU)

Reproduces the experimental design of *"How Emotion Shapes the Behavior of LLMs and Agents: A Mechanistic Study"*
(arXiv 2604.00005) as closely as the available resources allow. **This is not the small local proof-of-concept**
(no Qwen2.5-0.5B, no regression directions, no 5-question logic set). Both model paths are kept: the **hardware check
picks the model** — the paper's **Qwen3-8B** when the GPU can hold it (~21 GB), otherwise the smaller **Qwen3-1.7B**
(a clearly labeled fallback that is *not* the paper's model; both use the same code and differ only in configuration).

**What this notebook does** (each stage is resumable; re-running a cell skips finished work):
1. hardware check → 2. settings & acknowledgements → 3. choose the model tier from the hardware → 4. preflight (resources reachable?) → 5. load the model →
5. SAE + contrastive VAD feature identification (Sec. 4.2) → 6. α calibration → 7. LLM-task sweeps over δ ∈ {−9…+9}
(Sec. 5.2–5.4) → 8. latent validation (Sec. 5.6) → 9. agent (Sec. 5.5, optional, expensive) → 10. report + comparison with the paper's numbers.

**Needs** a GPU (≥ ~21 GB for Qwen3-8B, ≥ ~7–11 GB for the Qwen3-1.7B fallback), ~20 GB of storage for the 8B cache (~6 GB for
1.7B), and this notebook placed inside the project's `faithful/` folder (it imports the `esteer` package next to it).
Nothing here has been run on either real model yet — read the table below before trusting any number it produces.

## What is changed compared with the paper

**Substitutions and reconstructions** — each must be acknowledged in the settings cell (`ACK`), otherwise the run stops:

| # | Paper | This notebook | Why |
|---|---|---|---|
| 1 | "a trained SAE" at layer 17 — source, width, sparsity, training data **not given, checkpoint not released** | **Qwen-Scope** `Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_50`, layer 17 (TopK-50, 64K latents) | the paper's SAE does not exist publicly; this is the closest official one. It was trained on **Qwen3-8B-Base** but steers the post-trained Qwen3-8B, and was released *after* the paper |
| 2 | α in Eq. 7 — **value never given** | calibrated: largest α in a grid whose answer-validity rate at \|δ\|=9 stays ≥ 0.9 × unsteered, on LogiQA items disjoint from the evaluation set (or pin `ALPHA`) | needed to run at all; this is *my* rule, saved to `calibration.json` |
| 3 | subjective / safety outputs "evaluated by the LLM" — **judge not named** | the **unsteered backbone model** (Qwen3-8B, or Qwen3-1.7B on the small tier) scores them (1–5 rubric; harmful/biased/hallucination/refusal flags) | no external judge available on DSMLP; self-judging may bias scores |
| 4 | subsets "sampled proportionally", **sizes not given** | LogiQA 300, HumanEval 164 (all), MATH 350 (stratified by level), TinyStories 100, HarmBench 200 (all *standard* behaviors), HotpotQA 100, CAMEL-science 45 | budget; any n below the full dataset is a subset |
| 5 | agent: planner / decider / executor (prompts, tools, confidence **not given**) | prompts, JSON protocol, confidence elicitation, candidate generation are **mine**; steering applied to every call; **no external tools** | paper gives only the idea |
| 6 | label → VAD coordinates in [−10, 10] (**source not stated**) | NRC VAD Lexicon v2 ×10 if you provide the file (pairs matched on the other two dimensions); otherwise **hand-picked** high/low pole words | the lexicon is licensed and cannot be bundled |
| 7 | GAIA in the agent benchmark | optional, off by default (gated; text-only questions, no tools) | needs your HF token + tools |
| 8 | **Qwen3-8B** | Qwen3-8B when the GPU fits it. **Otherwise Qwen3-1.7B** + Qwen-Scope `SAE-Res-Qwen3-1.7B-Base-W32K-L0_50` at **layer 13** (= round(17/36 × 28), same relative depth) | an 8B model needs ~21 GB. The 1.7B model is much weaker: baselines drop and the paper's effects may shrink or vanish, so its numbers are **not comparable** with the paper's. No 4B tier: Qwen-Scope publishes no 4B SAE |

**Interpretations** (paper silent or ambiguous; not blocked, but my choice):
- **Layer**: output of `model.model.layers[17]` (0-indexed). The feature stage prints SAE reconstruction at layers 16/17/18 so an off-by-one shows up.
- **Eq. 6 sign**: read literally, `d/‖d‖·δ/δ_max` makes +δ and −δ steer **identically** (the sign cancels). I use the sign-preserving form (scale `δ/δ_max`); `eq6_variant="literal"` reproduces the literal reading.
- `‖h_k‖` is the per-token hidden-state norm; `δ_max = 9` (largest swept value); the decoder is linear, so Eq. 5 reduces to `W_dec·û` (unit-tested against the literal formula).
- Feature identification details: prefix *"Right now you are feeling ⟨word⟩."*; SAE latents averaged over task-description tokens; per (pair, task) top-50 by |Δ|; keep latents with a consistent sign in ≥ 50 % of rows.
- Qwen3 **thinking mode off**; per-task max generation lengths (LogiQA 512, HumanEval 768, MATH 1024, TinyStories 256, HarmBench 384) are my choice.
- **Precision**: bf16 on GPUs with native support (compute capability ≥ 8.0); otherwise fp32 if it fits, else fp16 (Pascal/Volta/Turing cards have no native bf16; Qwen can overflow in fp16).
- Metrics: AVR for subjective/safety tasks is my definition (completed story; substantive non-refusal); TSR counts invalid answers as wrong (`TSR_valid` is also saved).

**Same as the paper (on the 8B tier):** Qwen3-8B (bf16, greedy decoding) · top-50 contrastive latent selection per V/A/D dimension · additive hook at block output (Eq. 5–7) · each dimension swept over −9…+9 in steps of 3 with the others neutral · LogiQA 2.0 / HumanEval (pass@1) / MATH / TinyStories / HarmBench · AVR, TSR, judge scores, safety-risk probabilities, fluctuation range · Sec. 5.6 original vs random vs half-replaced latents.

**Not implemented:** Appendix A.1 (trained VAD analyzer / linear-controllability table) and A.2–A.4 (other datasets, sampling robustness, gpt-oss-20B).

**Other data caveats:** the LogiQA 2.0 mirror interleaves NLI-style and Chinese rows with the multiple-choice items; only the **1,572 English multiple-choice test items** are used. HarmBench comes from the public GitHub CSV (the HF copy is gated).

In [ ]:
# --- Hardware check ---------------------------------------------------------------------------
import os
os.environ.setdefault("USE_TF", "0")                    # keep transformers from probing TensorFlow (it deadlocked on one machine)
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # macOS/conda duplicate-OpenMP workaround; harmless elsewhere

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("CPU mode: activation extraction may be slow, and you need enough system RAM.")

# Extra info for this experiment. The model is chosen from this hardware in the "Choose the model tier" cell below.
try:
    RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    print("System RAM visible (GB):", round(RAM_GB, 1), "(weights are loaded to CPU first; ~32 GB recommended for 8B)")
except (ValueError, OSError, AttributeError):
    RAM_GB = None
if DEVICE == "cuda":
    cc = torch.cuda.get_device_capability(0)
    print(f"Compute capability: {cc[0]}.{cc[1]}", "(native bf16)" if cc[0] >= 8 else "(no native bf16: will use fp32/fp16)")
    print("Next: the hardware picks Qwen3-8B (needs ~21 GB) or the smaller Qwen3-1.7B fallback.")
else:
    print("No GPU: neither Qwen3 tier is usable (only the tiny 'smoke' profile can run on CPU).")

## Settings and acknowledgements

Edit this cell, then run it. **`ACK` is your explicit acceptance of the substitutions in the table above** (`smaller-model` = you accept the
Qwen3-1.7B fallback when the GPU is too small) — remove an item to refuse that substitution (the run will stop and say what it needs). Everything is resumable: set `RUN_ID` to an existing
run id to continue after a restart or the 12-hour DSMLP limit.

`MODEL_TIER="auto"` lets the hardware decide; force `"qwen3-8b"` (the run stops if it does not fit) or `"qwen3-1.7b"` to override.

Profiles (see `configs/`): `full` = all 19 conditions with the default sample sizes · `pilot` = real model, small samples,
13 conditions (**start here**) · `smoke` = plumbing test with a tiny model and a *random* SAE (not the paper).
Rough budget, upper bound and *assumed* throughput: pilot ≈ 2–5 GPU-h, full ≈ 11–28 GPU-h (+ agent). At DSMLP's default of
10 SU/week, `medium` costs 1 SU/GPU-h — `full` does not fit in one week at the full rate.

In [ ]:
PROFILE = "pilot"            # "pilot" first; "full" for the complete sweep; "smoke" = plumbing only
RUN_ID = ""                  # "" = new run (<profile>-<timestamp>); set an existing id to RESUME
STAGES = ["preflight", "features", "calibrate", "llm", "validation", "report"]   # add "agent" (most expensive) when ready
OUT_ROOT = "results_faithful"                 # outputs go to <OUT_ROOT>/<RUN_ID>/ (never next to the local PoC results)
HF_HOME_DIR = ""             # e.g. "/home/<user>/private/hf_cache" to move the ~20 GB model cache; "" = default
NRC_VAD_PATH = None          # path to NRC-VAD-Lexicon v2 text file (https://saifmohammad.com/WebPages/nrc-vad.html); None = hand-picked words
ALPHA = None                 # None = calibrate (stage "calibrate"); or pin a float
MODEL_TIER = "auto"          # "auto" = pick by GPU (Qwen3-8B if it fits, else Qwen3-1.7B) | "qwen3-8b" | "qwen3-1.7b" | "none"

# Acknowledged substitutions (numbers refer to the table above). Delete an item to refuse it.
ACK = [
    "sae-qwen-scope",        # 1  Qwen-Scope SAE instead of the paper's unreleased SAE
    "alpha",                 # 2  alpha chosen by calibration rule / manual value
    "judge-backbone",        # 3  the unsteered backbone model as judge
    "subset-sizes",          # 4  sampled subsets, sizes are mine
    "smaller-model",         # 8  accept the Qwen3-1.7B fallback if the GPU cannot hold Qwen3-8B (unused if 8B is chosen)
]
if NRC_VAD_PATH is None:
    ACK.append("label-handpicked")        # 6  hand-picked emotion words instead of NRC-matched pairs
if "agent" in STAGES:
    ACK.append("agent-reconstruction")    # 5  reconstructed agent
if PROFILE == "smoke":
    ACK.append("random-sae-smoke-only")   # untrained random SAE: results meaningless

if HF_HOME_DIR:
    os.environ["HF_HOME"] = HF_HOME_DIR   # must be set before transformers is imported
print("Profile:", PROFILE, "| stages:", STAGES)
print("Acknowledged:", ACK)

In [ ]:
# Locate the esteer package (this notebook lives in the project's faithful/ folder) and check dependencies.
import sys, importlib
here = Path.cwd().resolve()
FAITHFUL_DIR = next((p for p in [here, *here.parents, *(c for c in here.glob("*/faithful") if c.is_dir())]
                     if (p / "esteer" / "__init__.py").exists()), None)
if FAITHFUL_DIR is None:
    raise RuntimeError("Cannot find the esteer/ package. Open this notebook from <project>/faithful/ "
                       "(copy the whole project to the machine, e.g. with rsync).")
sys.path.insert(0, str(FAITHFUL_DIR))
print("faithful dir:", FAITHFUL_DIR)
MODEL_STAGES_SELECTED = bool({"features", "calibrate", "llm", "validation", "agent"} & set(STAGES))

missing = []
for mod, minimum in [("transformers", "4.51.0"), ("datasets", None), ("accelerate", None), ("huggingface_hub", None)]:
    try:
        v = importlib.import_module(mod).__version__
        print(f"{mod} {v}")
        if minimum and tuple(int(x) for x in v.split(".")[:2]) < tuple(int(x) for x in minimum.split(".")[:2]):
            missing.append(f"{mod}>={minimum} (found {v}; Qwen3 needs >= {minimum})")
    except ImportError:
        missing.append(mod)
if missing:
    raise RuntimeError(
        "Missing/too-old packages: " + ", ".join(missing) + "\n"
        "Install from the project's requirements, preferably into a venv with its own kernel (DSMLP docs: "
        "https://dsmlp-docs.ucsd.edu/environments/customizing-your-environment/), e.g.\n"
        f"    pip install -r {FAITHFUL_DIR / 'requirements.txt'}\n"
        "Do not reinstall torch on DSMLP: the container ships a CUDA-matched build.")

## Choose the model tier from the hardware

Detects GPU 0, evaluates each tier (`qwen3-8b` first, then `qwen3-1.7b`) in the dtype the card supports, and picks the first that
fits with headroom for the SAE, activations and KV cache. It also sizes the generation batch. Nothing is silently downgraded:
a forced tier that does not fit stops the run, and a smaller tier needs the `smaller-model` acknowledgement.

In [ ]:
from esteer.hardware import resolve
import json as _json

_profile_path = FAITHFUL_DIR / "configs" / f"{PROFILE}.json"
_pins_model = "model_name" in _json.loads(_profile_path.read_text())      # e.g. the smoke profile names its own tiny model
TIER, TIER_KW, HW, REC = resolve(MODEL_TIER, _pins_model, MODEL_STAGES_SELECTED, log=print)
if REC is not None:
    display(pd.DataFrame(REC.candidates))
    print("chosen:", TIER, TIER_KW)
else:
    print("profile pins its own model; no tier selection" + (" (smoke/plumbing run - not the paper)" if PROFILE == "smoke" else ""))

In [ ]:
from IPython.display import Image, Markdown, display

from esteer.config import load_config
from esteer.fidelity import MATRIX, check_acks, required_acks
from esteer.run_experiment import MODEL_STAGES, get_alpha, load_directions, make_logger, stage_features
from esteer.resources import blocked, estimate_cost, run_preflight

overrides = [] if ALPHA is None else [f"alpha={ALPHA}"]
cfg = load_config(str(FAITHFUL_DIR / "configs" / f"{PROFILE}.json"), overrides, model_tier=TIER,
                  run_id=RUN_ID or None, out_root=OUT_ROOT, nrc_vad_path=NRC_VAD_PATH, ack=list(ACK), **TIER_KW)
RUN_DIR = Path(cfg.run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)
log = make_logger(str(RUN_DIR / "run.log"))

if MODEL_STAGES & set(STAGES):
    check_acks(cfg, STAGES, lexicon_available=bool(cfg.nrc_vad_path))   # stops with an explanation if something is unacknowledged
(RUN_DIR / "config.json").write_text(json.dumps(cfg.to_dict(), indent=2))
(RUN_DIR / "hardware.json").write_text(_json.dumps({"hardware": HW, "requested_tier": MODEL_TIER, "chosen_tier": TIER,
                                                    "recommendation": None if REC is None else REC.__dict__}, indent=2, default=str))
(RUN_DIR / "fidelity.json").write_text(json.dumps({"matrix": MATRIX, "acknowledged": cfg.ack,
                                                    "required": required_acks(cfg, STAGES, bool(cfg.nrc_vad_path))}, indent=2))
torch.manual_seed(cfg.seed)

from esteer.evaluate import conditions
summary = pd.DataFrame([
    ("run dir", str(RUN_DIR.resolve())), ("profile", cfg.profile), ("model tier", cfg.model_tier or "(profile-defined)"), ("model", f"{cfg.model_name} ({cfg.model_dtype})"),
    ("batch size", cfg.batch_size),
    ("steering layer", f"model.model.layers[{cfg.layer}] output"), ("SAE", f"{cfg.sae_source}: {cfg.sae_repo}  (k={cfg.sae_k})"),
    ("deltas", cfg.deltas), ("delta_max / Eq.6", f"{cfg.delta_max} / {cfg.eq6_variant}"), ("conditions", len(conditions(cfg))),
    ("alpha", "calibrate" if cfg.alpha is None else cfg.alpha),
    ("LLM tasks (n)", cfg.tasks), ("max_new_tokens", cfg.max_new_tokens), ("thinking mode", cfg.enable_thinking),
    ("agent datasets (n)", cfg.agent_datasets if "agent" in STAGES else "(agent stage not selected)"),
], columns=["setting", "value"])
display(summary)
display(pd.DataFrame(MATRIX)[["id", "status", "paper", "impl"]])        # full fidelity matrix, saved with the run

## Preflight — are the model, SAE, datasets and network reachable? (also prints the cost estimate)

In [ ]:
if "preflight" in STAGES:
    log("preflight:")
    pre = run_preflight(cfg, log)
    display(pd.DataFrame(pre["items"]))
    display(pd.Series(estimate_cost(cfg)).to_frame("estimate (upper bound; throughput ASSUMED, not measured)"))
    bl = blocked(pre)
    if bl and (MODEL_STAGES & set(STAGES)):
        raise RuntimeError("Preflight found blockers:\n" + "\n".join(f"- {i['name']}: {i['detail']}" for i in bl))

## Load the model

In [ ]:
import time
bundle = None
if MODEL_STAGES & set(STAGES):
    from esteer.model import ModelBundle
    t0 = time.time()
    log(f"loading {cfg.model_name} ({cfg.model_dtype}) ...")
    bundle = ModelBundle(cfg)
    log(f"loaded on {bundle.device} in {time.time() - t0:.0f}s; {bundle.n_layers} layers")
    if bundle.device == "cuda":
        print("GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

## Sec. 4.2 — SAE and VAD feature identification

Loads the SAE, checks that it reconstructs the hooked layer best (a peak away from `cfg.layer` means an index offset),
builds contrastive pairs (same task description, only the emotion word changes), and selects stable latents per dimension.
Afterwards the SAE is freed from GPU memory. **Check the direction cosines**: the local PoC's regression directions were
near-collinear (|cos| > 0.98), which made valence/arousal/dominance behave identically.

In [ ]:
if "features" in STAGES:
    log("stage: features")
    stage_features(bundle, cfg, str(RUN_DIR), log)

FEATS_PATH = RUN_DIR / "features" / "features.json"
if FEATS_PATH.exists():
    feats = json.loads(FEATS_PATH.read_text())
    chk = json.loads((RUN_DIR / "features" / "sae_check.json").read_text())
    if cfg.sae_source == "random":
        print("WARNING: untrained random SAE - plumbing test only, nothing below is meaningful.")
    display(pd.DataFrame([{
        "dimension": d, "stable latents": len(f["indices"]), "rows (pairs x tasks)": f["n_rows"],
        "threshold relaxed": f["relaxed_threshold"], "example pairs (+/-)": ", ".join(f"{p['pos']}/{p['neg']}" for p in f["pairs"][:4])}
        for d, f in feats["dims"].items()]))
    print("label source:", feats["label_source"], "| direction cosines:", feats["diagnostics"]["cosine"],
          "| shared latents:", feats["diagnostics"]["shared_latents"])
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].bar(list(chk["fve_by_layer"]), list(chk["fve_by_layer"].values()))
    ax[0].set(title=f"SAE reconstruction (FVE) by hooked layer; configured = {cfg.layer}", xlabel="layer", ylabel="FVE")
    for d, f in feats["dims"].items():
        ax[1].hist(f["stability"], bins=10, alpha=0.5, label=d)
    ax[1].set(title="stability of selected latents", xlabel="fraction of rows with consistent sign"); ax[1].legend()
    plt.tight_layout(); plt.show()

## α calibration (α in Eq. 7 is not given in the paper)

Rule (mine, see the table): the largest α in the grid whose answer-validity rate at |δ| = 9 on every dimension and sign stays
≥ 0.9 × the unsteered rate, evaluated on LogiQA items **disjoint from the reported sample**. If no α qualifies, the smallest
grid value is used and a warning is printed — inspect the table before trusting the sweep.

In [ ]:
from esteer.calibrate import calibrate_alpha
from esteer.steering import EmotionSteering

DIRS = load_directions(str(RUN_DIR)) if (RUN_DIR / "features" / "directions.pt").exists() else None
if "calibrate" in STAGES:
    if cfg.alpha is not None:
        log(f"alpha fixed by config: {cfg.alpha} (calibration skipped)")
    elif DIRS is None:
        raise RuntimeError("Run the 'features' stage first (directions.pt is missing).")
    elif not (RUN_DIR / "calibration.json").exists():
        log("stage: calibrate (alpha)")
        calibrate_alpha(bundle, cfg, DIRS, str(RUN_DIR), log)
if (RUN_DIR / "calibration.json").exists():
    cal = json.loads((RUN_DIR / "calibration.json").read_text())
    display(pd.DataFrame([{"alpha": r["alpha"], "min AVR @ |delta|=9": r["min_AVR"]} for r in cal["rows"]]))
    print(f"unsteered AVR = {cal['neutral_AVR']:.3f}; chosen alpha = {cal['chosen_alpha']}  ({cal['rule']})",
          "| NO ALPHA MET THE FLOOR" if cal["no_alpha_met_floor"] else "")

def arm():
    """Attach the Eq. 5-7 steering (directions + alpha) to the model bundle."""
    alpha = get_alpha(cfg, str(RUN_DIR))
    bundle.steering = EmotionSteering(DIRS, alpha, cfg.delta_max, cfg.eq6_variant)
    return alpha

## Sec. 5.2–5.4 — LLM tasks over the VAD sweep

LogiQA 2.0, HumanEval (pass@1; **executes model-generated code in subprocesses**), MATH, TinyStories (judge-scored) and
HarmBench (judge-scored). Each (task, condition) is saved as soon as it finishes, so a restart skips completed work.
HarmBench generations may contain harmful text; they stay in the run directory only.

In [ ]:
from esteer.evaluate import load_summaries, run_llm_stage

if "llm" in STAGES:
    log(f"stage: llm (alpha={arm()})")
    run_llm_stage(bundle, cfg, str(RUN_DIR), log)
    print(f"measured generation throughput: {bundle.throughput():.0f} tok/s ({bundle.tokens_generated:,} tokens)")

from esteer.report import _plot_task
S = load_summaries(str(RUN_DIR))
FIG_DIR = RUN_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
for task, conds in S.items():
    p = _plot_task(conds, task, str(FIG_DIR))
    if p:
        display(Markdown(f"**{task}** (n={next(iter(conds.values()))['summary']['n']} per condition, {len(conds)} conditions done)"))
        display(Image(p))

## Sec. 5.6 — are the selected latents really VAD features? (original vs random vs half-replaced, LogiQA)

In [ ]:
if "validation" in STAGES:
    from esteer.calibrate import run_validation
    from esteer.run_experiment import load_sae
    log("stage: validation (Sec. 5.6)")
    sae_cpu = load_sae(cfg, "cpu")                       # only W_dec is needed here
    run_validation(bundle, sae_cpu, json.loads(FEATS_PATH.read_text()), cfg, get_alpha(cfg, str(RUN_DIR)), str(RUN_DIR), log)
    del sae_cpu

from esteer.report import _curve
vroot = RUN_DIR / "validation"
if vroot.exists():
    fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
    for variant in sorted(p.name for p in vroot.iterdir()):
        vs = load_summaries(str(RUN_DIR), f"validation/{variant}").get("logiqa", {})
        for ax, dim in zip(axes, ["valence", "arousal", "dominance"]):
            pts = [(d, v) for d, v in _curve(vs, dim, "TSR") if v is not None]
            if pts:
                ax.plot(*zip(*pts), marker="o", label=variant)
            ax.set_title(dim); ax.set_xlabel("delta"); ax.grid(alpha=.3)
    axes[0].set_ylabel("LogiQA TSR"); axes[0].legend(); plt.tight_layout(); plt.show()

## Sec. 5.5 — agent (optional, most expensive stage)

Planner / decider / executor agent on HotpotQA and CAMEL-science; **reconstructed** from the paper's description (see the
table). Add `"agent"` to `STAGES` and run this cell last; it can be resumed like every other stage.

In [ ]:
if "agent" in STAGES:
    from esteer.agent import run_agent_stage
    log(f"stage: agent (alpha={arm()})")
    run_agent_stage(bundle, cfg, str(RUN_DIR), log)
else:
    print('Agent stage not selected (add "agent" to STAGES to run it).')

## Report — tables, figures and the paper's reported numbers next to this run's

In [ ]:
from esteer.report import _fmt, build_report, paper_comparison

if "report" in STAGES:
    build_report(str(RUN_DIR), log)
S = load_summaries(str(RUN_DIR))
rows = paper_comparison(S)
if rows:
    display(Markdown("**Paper's quoted numbers vs this run.** Orientation only: different SAE, sample sizes and judge; "
                     "with n in the tens, a few points are noise. A match is not proof of reproduction and a mismatch is not by itself a failure."))
    display(pd.DataFrame([{"quantity": r["claim"], "paper": r["paper"], "this run": _fmt(r["ours"]), "note": r["note"]} for r in rows]))
print("\nOutputs:", RUN_DIR.resolve())
for p in sorted(RUN_DIR.glob("*")):
    print("  ", p.name)

## Where things are saved, and how to get them back

`results_faithful/<run_id>/` next to this notebook (or `OUT_ROOT`): `REPORT.md`, `results_long.csv`, `figures/`,
`config.json`, `fidelity.json`, `resources_report.json`, `calibration.json`, `features/`, `llm/…` (every generation, parsed
answer and judge output), `validation/…`, `agent/…`, `run.log`. This folder is git-ignored.

On DSMLP, copy it to your laptop with (from the project folder on your laptop):
`bash dsmlp/fetch_faithful_results.sh <run_id>` — or `rsync -av <user>@dsmlp-login.ucsd.edu:<project>/results_faithful/<run_id>/ .`

To **resume** after a kernel restart or the runtime limit: set `RUN_ID` to this run's id in the settings cell and re-run from the top.
For long unattended runs prefer the batch job in `../dsmlp/submit_faithful.sh`, which executes the same code.